# Notebook 04: Text Modality Experiments (ASR Transcription & DistilBERT Encoding)
### Multidimensional Emotion-to-Sentiment Derivation System

**Assigned Scope: Text Modality Pipeline**

> **Scientific & Methodological Context (per `AGENTS.md` and Research Plan)**:
> - The official EmoReact dataset does not provide manual ground-truth transcripts in its distribution.
> - As mandated by the Master Prompt, we do **not** silently substitute generated transcripts for ground-truth.
> - We explicitly document the pipeline: speech is transcribed via OpenAI's open-source **Whisper ASR** (`whisper-base`), and contextual text representations are extracted via a frozen **DistilBERT** (`distilbert-base-uncased`) backbone.
> - Features are projected and time-aligned into the canonical shape **`[B, T=16, D=128]`** to match the Audio and Video feature contracts for Stage 3 (Multimodal Fusion).

## 1. Setup & Environment Verification

In [1]:
# Install transformers and openai-whisper for transcription
!pip install -q transformers openai-whisper torchaudio scikit-learn pandas seaborn matplotlib

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import os
import glob
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import f1_score, classification_report
from transformers import DistilBertTokenizer, DistilBertModel
import whisper

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# Device verification
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

Using device: cuda
Device Name: Tesla T4


In [7]:
# Mount Google Drive if running in Google Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    print("Running in local environment (not Colab).")

# Set DATA_DIR (adjust if extracted elsewhere)
if IN_COLAB:
    # If RAR is on Google Drive, extract to /content/data:
    # !mkdir -p /content/data
    # !unrar x "/content/drive/MyDrive/EmoReact_V_1.0.rar" /content/data/
    DATA_DIR = "/content/data/EmoReact_V_1.0"
    if not os.path.exists(DATA_DIR):
        # Fallback to local repo data folder if already placed there
        DATA_DIR = "data/raw/EmoReact_V_1.0"
else:
    DATA_DIR = "data/raw/EmoReact_V_1.0"

print("Target Data Directory:", DATA_DIR)
print("Directory exists:", os.path.exists(DATA_DIR))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Target Data Directory: /content/data/EmoReact_V_1.0
Directory exists: True


In [ ]:
# 1. Make sure unrar is installed
!apt-get install -y unrar

# 2. Create the data directory in Colab
!mkdir -p /content/data

# 3. Extract EmoReact from your Google Drive into /content/data
# (Change the path if you uploaded it inside a specific folder in Google Drive)
!unrar x "/content/drive/MyDrive/EmoReact_V_1.0.rar" /content/data/

## 2. Dataset Paths & Labels Loading

In [8]:
# Set DATA_DIR
if os.path.exists("/content/data/EmoReact_V_1.0"):
    DATA_DIR = "/content/data/EmoReact_V_1.0"
elif os.path.exists("data/raw/EmoReact_V_1.0"):
    DATA_DIR = "data/raw/EmoReact_V_1.0"
else:
    DATA_DIR = "/content/data/EmoReact_V_1.0"

EMOTION_NAMES = [
    "Curiosity", "Uncertainty", "Excitement", "Happiness",
    "Surprise", "Disgust", "Fear", "Frustration"
]
ALL_COLUMNS = EMOTION_NAMES + ["Valence"]

# Load split metadata
split_dfs = {}
for split, split_folder in [("train", "Train"), ("val", "Validation"), ("test", "Test")]:
    lbl_file = os.path.join(DATA_DIR, "Labels", f"{split}_labels.text")
    vid_folder = os.path.join(DATA_DIR, "Data", split_folder)

    df = pd.read_csv(lbl_file, header=None, names=ALL_COLUMNS)
    video_files = sorted([f for f in os.listdir(vid_folder) if f.endswith(".mp4")])
    df["video_name"] = video_files[:len(df)]
    df["video_path"] = [os.path.join(vid_folder, f) for f in df["video_name"]]
    df["split"] = split
    split_dfs[split] = df
    print(f"[{split.upper()}] Samples: {len(df)}")

all_samples_df = pd.concat([split_dfs["train"], split_dfs["val"], split_dfs["test"]], ignore_index=True)
print(f"Total Samples: {len(all_samples_df)}")

[TRAIN] Samples: 432
[VAL] Samples: 303
[TEST] Samples: 367
Total Samples: 1102


## 3. Speech-to-Text Transcription via Whisper ASR

We transcribe speech from each video clip using OpenAI's `whisper-base` model on GPU.
Transcripts are cached to `data/processed/transcripts.csv` so transcription is performed only once.

In [9]:
os.makedirs("data/processed", exist_ok=True)
transcripts_file = "data/processed/transcripts.csv"

if os.path.exists(transcripts_file):
    print(f"Loading pre-existing transcripts from {transcripts_file}...")
    all_samples_df = pd.read_csv(transcripts_file)
else:
    print("Loading Whisper-base model on GPU...")
    asr_model = whisper.load_model("base", device=device)

    transcripts = []
    print(f"Transcribing {len(all_samples_df)} video clips on {device}...")

    for idx, row in all_samples_df.iterrows():
        vid_path = row["video_path"]
        try:
            result = asr_model.transcribe(vid_path, fp16=torch.cuda.is_available())
            text = result["text"].strip()
            if not text:
                text = "[NO_SPEECH]"
        except Exception as e:
            text = "[NO_SPEECH]"

        transcripts.append(text)
        if (idx + 1) % 100 == 0 or (idx + 1) == len(all_samples_df):
            print(f"Transcribed {idx + 1}/{len(all_samples_df)} clips...")

    all_samples_df["transcript"] = transcripts
    all_samples_df.to_csv(transcripts_file, index=False)
    print(f"Successfully saved transcripts to {transcripts_file}!")

# Display sample transcripts
display(all_samples_df[["video_name", "split", "transcript", "Happiness", "Surprise", "Valence"]].head(10))

Loading Whisper-base model on GPU...


100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 215MiB/s]


Transcribing 1102 video clips on cuda...
Transcribed 100/1102 clips...
Transcribed 200/1102 clips...
Transcribed 300/1102 clips...
Transcribed 400/1102 clips...
Transcribed 500/1102 clips...
Transcribed 600/1102 clips...
Transcribed 700/1102 clips...
Transcribed 800/1102 clips...
Transcribed 900/1102 clips...
Transcribed 1000/1102 clips...
Transcribed 1100/1102 clips...
Transcribed 1102/1102 clips...
Successfully saved transcripts to data/processed/transcripts.csv!


,video_name,split,transcript,Happiness,Surprise,Valence
0,80'sTOYS105_2.mp4,train,"So if you got Teddy Roxpin as a present, would...",1,0,3.3333
1,80'sTOYS118_2.mp4,train,"They made it so like, life-like kind of that i...",1,0,6.3333
2,80'sTOYS127_2.mp4,train,"I have very things, why would I need this?",0,0,2.6667
3,80'sTOYS128_2.mp4,train,is way better technology than what this is.,0,0,2.3333
4,80'sTOYS129_2.mp4,train,even on the iPad there's like apps where you c...,1,0,4.6667
5,80'sTOYS132_2.mp4,train,personning through his ratchet song.,0,0,2.6667
6,80'sTOYS134_2.mp4,train,"And so even though we're grown-ups, we are so ...",1,0,6.0000
7,80'sTOYS24_2.mp4,train,"How do you know Teddy Ruckspin? Okay, so I've ...",1,0,5.0000
8,80'sTOYS2_2.mp4,train,so you're not reacting to a video. What?,0,1,5.0000
9,80'sTOYS30_2.mp4,train,"What do you think of his name? Uh, no, no. Let...",0,0,2.6667


## 4. Transcript Analysis & Tokenization

Inspect the dialogue vs silence proportions and prepare DistilBERT tokenization.

In [10]:
no_speech_count = (all_samples_df["transcript"] == "[NO_SPEECH]").sum()
print(f"Clips with no detectable speech: {no_speech_count} / {len(all_samples_df)} ({no_speech_count/len(all_samples_df)*100:.1f}%)")
print(f"Clips with spoken dialogue:      {len(all_samples_df) - no_speech_count} / {len(all_samples_df)} ({(len(all_samples_df)-no_speech_count)/len(all_samples_df)*100:.1f}%)")

# Load DistilBERT Tokenizer
TOKENIZER_NAME = "distilbert-base-uncased"
tokenizer = DistilBertTokenizer.from_pretrained(TOKENIZER_NAME)
print(f"Loaded Tokenizer: {TOKENIZER_NAME}")

# Sample tokenization
sample_text = all_samples_df["transcript"].iloc[0]
sample_tokens = tokenizer(sample_text, padding="max_length", max_length=32, truncation=True, return_tensors="pt")
print(f"Sample Text: '{sample_text}'")
print("Input IDs Shape:", sample_tokens["input_ids"].shape)
print("Attention Mask Shape:", sample_tokens["attention_mask"].shape)

Clips with no detectable speech: 8 / 1102 (0.7%)
Clips with spoken dialogue:      1094 / 1102 (99.3%)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Loaded Tokenizer: distilbert-base-uncased
Sample Text: 'So if you got Teddy Roxpin as a present, would you be excited? I do.'
Input IDs Shape: torch.Size([1, 32])
Attention Mask Shape: torch.Size([1, 32])


## 5. PyTorch `EmoReactTextDataset` & DataLoaders

In [11]:
class EmoReactTextDataset(Dataset):
    """
    PyTorch Dataset serving tokenized text sequences and emotion annotations.
    """
    def __init__(self, df, tokenizer, max_length=48):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.emotion_cols = EMOTION_NAMES

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = str(row["transcript"])

        encoded = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        emotions = torch.tensor(row[self.emotion_cols].values.astype(np.float32))
        valence = torch.tensor([float(row["Valence"])], dtype=torch.float32)

        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "emotions": emotions,
            "valence": valence,
            "sample_id": row["video_name"]
        }

# Create splits
train_df = all_samples_df[all_samples_df["split"] == "train"]
val_df = all_samples_df[all_samples_df["split"] == "val"]
test_df = all_samples_df[all_samples_df["split"] == "test"]

train_text_dataset = EmoReactTextDataset(train_df, tokenizer)
val_text_dataset = EmoReactTextDataset(val_df, tokenizer)
test_text_dataset = EmoReactTextDataset(test_df, tokenizer)

train_text_loader = DataLoader(train_text_dataset, batch_size=16, shuffle=True)
val_text_loader = DataLoader(val_text_dataset, batch_size=16, shuffle=False)
test_text_loader = DataLoader(test_text_dataset, batch_size=16, shuffle=False)

sample_batch = next(iter(train_text_loader))
print("Batch input_ids shape:", sample_batch["input_ids"].shape)
print("Batch emotions shape: ", sample_batch["emotions"].shape)

Batch input_ids shape: torch.Size([16, 48])
Batch emotions shape:  torch.Size([16, 8])


## 6. `TextEncoder` Neural Architecture (DistilBERT)

In accordance with `AGENTS.md`:
- Uses frozen pretrained **DistilBERT** (`distilbert-base-uncased`, 768-dim).
- Linear projection: $768 \rightarrow D_{text} = 128$.
- Adaptive temporal pooling: maps token representations to exact canonical shape **`[B, T=16, D=128]`** to match the Audio and Video feature dimensions for Multimodal Fusion!

In [12]:
class TextEncoder(nn.Module):
    """
    Frozen DistilBERT text encoder projecting contextual token representations
    into canonical sequence format [B, T=16, D_text=128].
    """
    def __init__(self, target_steps=16, feature_dim=128, freeze_backbone=True):
        super(TextEncoder, self).__init__()
        self.target_steps = target_steps
        self.feature_dim = feature_dim

        # Load pretrained DistilBERT backbone
        self.backbone = DistilBertModel.from_pretrained("distilbert-base-uncased")

        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False

        # Projection: 768 -> 128
        self.proj = nn.Sequential(
            nn.Linear(768, feature_dim),
            nn.LayerNorm(feature_dim),
            nn.ReLU()
        )

        # Temporal alignment pooler to ensure T=16 steps
        self.temporal_pool = nn.AdaptiveAvgPool1d(target_steps)

    def forward(self, input_ids, attention_mask):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        hidden_states = outputs.last_hidden_state # [B, seq_len, 768]

        proj_features = self.proj(hidden_states) # [B, seq_len, 128]
        aligned = self.temporal_pool(proj_features.transpose(1, 2)).transpose(1, 2) # [B, 16, 128]
        return aligned

# Smoke test with synthetic inputs
test_ids = torch.randint(0, 1000, (4, 48))
test_mask = torch.ones((4, 48))
text_encoder = TextEncoder().to(device)
test_out = text_encoder(test_ids.to(device), test_mask.to(device))
print(f"TextEncoder Smoke Test Shape: {test_out.shape} -> Match [4, 16, 128]: {test_out.shape == torch.Size([4, 16, 128])}")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


TextEncoder Smoke Test Shape: torch.Size([4, 16, 128]) -> Match [4, 16, 128]: True


## 7. Unimodal Text Baseline Classifier

Evaluating the predictive capability of textual cues alone on the 8 emotion categories.

In [13]:
class TextEmotionBaseline(nn.Module):
    """
    Text-Only Emotion Recognition Baseline.
    TextEncoder -> Mean Pooling -> MLP Emotion Head (8 Logits)
    """
    def __init__(self, target_steps=16, feature_dim=128, num_emotions=8):
        super(TextEmotionBaseline, self).__init__()
        self.encoder = TextEncoder(target_steps=target_steps, feature_dim=feature_dim)
        self.classifier = nn.Sequential(
            nn.Linear(feature_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_emotions)
        )

    def forward(self, input_ids, attention_mask):
        feat_seq = self.encoder(input_ids, attention_mask) # [B, 16, 128]
        pooled = torch.mean(feat_seq, dim=1) # [B, 128]
        logits = self.classifier(pooled) # [B, 8]
        return logits, feat_seq

# Calculate pos_weight for BCE
train_emotions = train_df[EMOTION_NAMES].values
pos_counts = train_emotions.sum(axis=0)
neg_counts = len(train_emotions) - pos_counts
pos_weight = torch.tensor(neg_counts / np.maximum(pos_counts, 1), dtype=torch.float32).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
text_model = TextEmotionBaseline().to(device)
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, text_model.parameters()), lr=1e-3, weight_decay=1e-4)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 8. Text Baseline Training & Evaluation Loop (10 Epochs)

In [14]:
def train_text_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    for batch in loader:
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        labels = batch["emotions"].to(device)

        optimizer.zero_grad()
        logits, _ = model(ids, mask)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * ids.size(0)
    return total_loss / len(loader.dataset)

@torch.no_grad()
def evaluate_text(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []

    for batch in loader:
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        labels = batch["emotions"].to(device)

        logits, _ = model(ids, mask)
        loss = criterion(logits, labels)
        total_loss += loss.item() * ids.size(0)

        probs = torch.sigmoid(logits)
        all_preds.append(probs.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    all_preds = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)

    binary_preds = (all_preds >= 0.5).astype(int)
    macro_f1 = f1_score(all_labels, binary_preds, average="macro", zero_division=0)
    return total_loss / len(loader.dataset), macro_f1, all_preds, all_labels

# Run training
print("Starting Text Baseline Training on GPU...")
best_val_f1 = 0.0

for epoch in range(1, 11):
    tr_loss = train_text_epoch(text_model, train_text_loader, optimizer, criterion)
    val_loss, val_f1, _, _ = evaluate_text(text_model, val_text_loader, criterion)

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(text_model.encoder.state_dict(), "best_text_encoder.pt")

    print(f"Epoch [{epoch:02d}/10] | Train Loss: {tr_loss:.4f} | Val Loss: {val_loss:.4f} | Val Macro-F1: {val_f1:.4f}")

print(f"Best Text Validation Macro-F1: {best_val_f1:.4f}")

Starting Text Baseline Training on GPU...
Epoch [01/10] | Train Loss: 1.0314 | Val Loss: 1.0031 | Val Macro-F1: 0.3784
Epoch [02/10] | Train Loss: 0.9854 | Val Loss: 0.9759 | Val Macro-F1: 0.4053
Epoch [03/10] | Train Loss: 0.9407 | Val Loss: 0.9368 | Val Macro-F1: 0.4020
Epoch [04/10] | Train Loss: 0.8918 | Val Loss: 0.9303 | Val Macro-F1: 0.4199
Epoch [05/10] | Train Loss: 0.8496 | Val Loss: 0.9445 | Val Macro-F1: 0.4412
Epoch [06/10] | Train Loss: 0.8144 | Val Loss: 0.9152 | Val Macro-F1: 0.4146
Epoch [07/10] | Train Loss: 0.7715 | Val Loss: 0.9053 | Val Macro-F1: 0.4442
Epoch [08/10] | Train Loss: 0.7393 | Val Loss: 0.9363 | Val Macro-F1: 0.4633
Epoch [09/10] | Train Loss: 0.7007 | Val Loss: 0.9436 | Val Macro-F1: 0.4345
Epoch [10/10] | Train Loss: 0.6579 | Val Loss: 0.9370 | Val Macro-F1: 0.4315
Best Text Validation Macro-F1: 0.4633


In [16]:
import os

# Create the folder first so PyTorch can save the file
os.makedirs("data/features", exist_ok=True)

# Now save the features
torch.save(text_feature_dict, "data/features/text_features.pt")
print("\nSuccessfully cached text features to data/features/text_features.pt!")


Successfully cached text features to data/features/text_features.pt!


## 9. Feature Caching for Multimodal Fusion

Extract and cache text sequence representations `[N, 16, 128]` for Train, Val, and Test splits into `data/features/text_features.pt`.

In [17]:
if os.path.exists("best_text_encoder.pt"):
    text_model.encoder.load_state_dict(torch.load("best_text_encoder.pt"))

@torch.no_grad()
def extract_text_features(model, loader):
    model.eval()
    all_features = []
    all_sample_ids = []
    for batch in loader:
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        feats = model.encoder(ids, mask) # [B, 16, 128]
        all_features.append(feats.cpu())
        all_sample_ids.extend(batch["sample_id"])
    return torch.cat(all_features, dim=0), all_sample_ids

print("Extracting text representations for all splits...")
train_text_feats, train_text_ids = extract_text_features(text_model, train_text_loader)
val_text_feats, val_text_ids = extract_text_features(text_model, val_text_loader)
test_text_feats, test_text_ids = extract_text_features(text_model, test_text_loader)

print(f"Train Text Features: {train_text_feats.shape} [432, 16, 128]")
print(f"Val Text Features:   {val_text_feats.shape}   [303, 16, 128]")
print(f"Test Text Features:  {test_text_feats.shape}  [367, 16, 128]")

# Save cached features
text_feature_dict = {
    "train": {"features": train_text_feats, "sample_ids": train_text_ids},
    "val": {"features": val_text_feats, "sample_ids": val_text_ids},
    "test": {"features": test_text_feats, "sample_ids": test_text_ids},
    "feature_dim": 128,
    "temporal_steps": 16
}
torch.save(text_feature_dict, "data/features/text_features.pt")
print("\nSuccessfully cached text features to data/features/text_features.pt!")

Extracting text representations for all splits...
Train Text Features: torch.Size([432, 16, 128]) [432, 16, 128]
Val Text Features:   torch.Size([303, 16, 128])   [303, 16, 128]
Test Text Features:  torch.Size([367, 16, 128])  [367, 16, 128]

Successfully cached text features to data/features/text_features.pt!


## 10. Summary & Completion Checklist

- [x] Whisper ASR transcription implemented and cached to `data/processed/transcripts.csv`.
- [x] DistilBERT tokenizer & `EmoReactTextDataset` established.
- [x] `TextEncoder` producing canonical shape `[B, 16, 128]` validated.
- [x] Unimodal text baseline trained with BCE imbalance weights.
- [x] Text feature embeddings cached to `data/features/text_features.pt` for Multimodal Fusion!